In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import f_oneway, kruskal
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0-8')
sns.set_palette("Set2")

In [ ]:
# Load cleaned data for all 5 countries
countries = ['ethiopia', 'kenya', 'sudan', 'tanzania', 'nigeria']
dfs = {}

for country in countries:
    file_path = f"../data/{country}_clean.csv"
    try:
        df = pd.read_csv(file_path)
        df['Country'] = country.capitalize()
        dfs[country] = df
        print(f"Loaded {country}: {len(df)} rows")
    except FileNotFoundError:
        print(f"File not found: {file_path}")

# Combine all into one dataframe
combined_df = pd.concat(dfs.values(), ignore_index=True)
print(f"\nCombined dataset shape: {combined_df.shape}")
print(f"Countries included: {combined_df['Country'].unique()}")

In [ ]:
# Ensure Date is datetime
if 'Date' in combined_df.columns:
    combined_df['Date'] = pd.to_datetime(combined_df['Date'])
    combined_df['Year'] = combined_df['Date'].dt.year
    combined_df['Month'] = combined_df['Date'].dt.month
    print("Date column processed successfully")
else:
    print("Date column not found. Creating from YEAR and DOY...")
    combined_df['Date'] = pd.to_datetime(combined_df['YEAR'] * 1000 + combined_df['DOY'], format="%Y%j")
    combined_df['Year'] = combined_df['Date'].dt.year
    combined_df['Month'] = combined_df['Date'].dt.month

In [ ]:
# Calculate monthly average temperature per country
monthly_temp = combined_df.groupby(['Country', 'Year', 'Month'])['T2M'].mean().reset_index()
monthly_temp['Date'] = pd.to_datetime(monthly_temp['Year'].astype(str) + '-' + monthly_temp['Month'].astype(str) + '-01')

# Plot
plt.figure(figsize=(14, 7))
for country in monthly_temp['Country'].unique():
    country_data = monthly_temp[monthly_temp['Country'] == country]
    plt.plot(country_data['Date'], country_data['T2M'], linewidth=1.5, label=country)

plt.title('Monthly Average Temperature Comparison (2015-2026)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/temperature_trend_comparison.png')
plt.show()

In [ ]:
# Calculate summary statistics for temperature
temp_summary = combined_df.groupby('Country')['T2M'].agg([
    ('Mean', 'mean'),
    ('Median', 'median'),
    ('Std Dev', 'std')
]).round(2)

# Add min and max
temp_summary['Min'] = combined_df.groupby('Country')['T2M'].min().round(2)
temp_summary['Max'] = combined_df.groupby('Country')['T2M'].max().round(2)

print("=== TEMPERATURE SUMMARY STATISTICS (°C) ===")
print(temp_summary)

In [ ]:
# Side-by-side boxplots for precipitation
plt.figure(figsize=(12, 6))
sns.boxplot(x='Country', y='PRECTOTCORR', data=combined_df)
plt.title('Precipitation Distribution by Country (2015-2026)', fontsize=14)
plt.xlabel('Country')
plt.ylabel('Daily Precipitation (mm/day)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../reports/precipitation_boxplots.png')
plt.show()

In [ ]:
# Calculate summary statistics for precipitation
precip_summary = combined_df.groupby('Country')['PRECTOTCORR'].agg([
    ('Mean', 'mean'),
    ('Median', 'median'),
    ('Std Dev', 'std')
]).round(2)

precip_summary['Min'] = combined_df.groupby('Country')['PRECTOTCORR'].min().round(2)
precip_summary['Max'] = combined_df.groupby('Country')['PRECTOTCORR'].max().round(2)

print("=== PRECIPITATION SUMMARY STATISTICS (mm/day) ===")
print(precip_summary)

In [ ]:
# Count extreme heat days per year per country
extreme_heat = combined_df.groupby(['Country', 'Year']).apply(
    lambda x: (x['T2M_MAX'] > 35).sum()
).reset_index(name='ExtremeHeatDays')

# Pivot for easier plotting
heat_pivot = extreme_heat.pivot(index='Year', columns='Country', values='ExtremeHeatDays').fillna(0)

# Plot
ax = heat_pivot.plot(kind='bar', figsize=(14, 6), width=0.8)
plt.title('Extreme Heat Days per Year (T2M_MAX > 35°C)', fontsize=14)
plt.xlabel('Year')
plt.ylabel('Number of Days')
plt.legend(title='Country', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.savefig('../reports/extreme_heat_days.png')
plt.show()

print("=== AVERAGE EXTREME HEAT DAYS PER YEAR ===")
print(extreme_heat.groupby('Country')['ExtremeHeatDays'].mean().sort_values(ascending=False).round(1))

In [ ]:
# Function to count consecutive dry days per year
def count_dry_spells(group):
    dry = (group['PRECTOTCORR'] < 1).astype(int)
    # Count max consecutive dry days
    max_dry = 0
    current = 0
    for val in dry:
        if val == 1:
            current += 1
            max_dry = max(max_dry, current)
        else:
            current = 0
    return max_dry

# Apply to each country-year
dry_spells = combined_df.groupby(['Country', 'Year']).apply(count_dry_spells).reset_index(name='MaxDrySpell')

# Plot
plt.figure(figsize=(14, 6))
sns.barplot(x='Year', y='MaxDrySpell', hue='Country', data=dry_spells)
plt.title('Maximum Consecutive Dry Days per Year', fontsize=14)
plt.xlabel('Year')
plt.ylabel('Days')
plt.legend(title='Country', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.savefig('../reports/dry_spells.png')
plt.show()

print("=== AVERAGE MAX DRY SPELL (DAYS) ===")
print(dry_spells.groupby('Country')['MaxDrySpell'].mean().sort_values(ascending=False).round(1))

In [ ]:
# Prepare data for ANOVA
country_groups = [combined_df[combined_df['Country'] == country]['T2M'].dropna() for country in combined_df['Country'].unique()]

# One-way ANOVA
f_stat, p_value = f_oneway(*country_groups)

print("=== ONE-WAY ANOVA FOR TEMPERATURE ===")
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4e}")

# Interpretation
alpha = 0.05
if p_value < alpha:
    print("\nConclusion: Reject null hypothesis. There are statistically significant temperature differences between countries.")
    print("This means the observed temperature variations across the five countries are real, not random.")
else:
    print("\nConclusion: Fail to reject null hypothesis. No statistically significant temperature differences found.")

In [ ]:
# Create vulnerability metrics
vulnerability_data = []

for country in combined_df['Country'].unique():
    country_data = combined_df[combined_df['Country'] == country]
    
    # Temperature trend (rate of change per year)
    yearly_temp = country_data.groupby('Year')['T2M'].mean()
    years = yearly_temp.index.values
    temps = yearly_temp.values
    slope = np.polyfit(years, temps, 1)[0]
    
    # Precipitation variability (coefficient of variation)
    precip_cv = country_data['PRECTOTCORR'].std() / country_data['PRECTOTCORR'].mean()
    
    # Extreme heat frequency
    heat_days = (country_data['T2M_MAX'] > 35).mean() * 365
    
    # Dry spell severity
    dry_spell = dry_spells[dry_spells['Country'] == country]['MaxDrySpell'].mean()
    
    vulnerability_data.append({
        'Country': country,
        'Warming Rate (°C/year)': round(slope, 4),
        'Precipitation CV': round(precip_cv, 2),
        'Extreme Heat Days/Year': round(heat_days, 1),
        'Max Dry Spell (days)': round(dry_spell, 1)
    })

vulnerability_df = pd.DataFrame(vulnerability_data)

# Create ranking (higher values = more vulnerable)
vulnerability_df['Warming Rank'] = vulnerability_df['Warming Rate (°C/year)'].rank(ascending=False)
vulnerability_df['Precip Rank'] = vulnerability_df['Precipitation CV'].rank(ascending=False)
vulnerability_df['Heat Rank'] = vulnerability_df['Extreme Heat Days/Year'].rank(ascending=False)
vulnerability_df['Dry Rank'] = vulnerability_df['Max Dry Spell (days)'].rank(ascending=False)

# Overall vulnerability score (lower rank = more vulnerable)
vulnerability_df['Vulnerability Score'] = (
    vulnerability_df['Warming Rank'] + 
    vulnerability_df['Precip Rank'] + 
    vulnerability_df['Heat Rank'] + 
    vulnerability_df['Dry Rank']
)
vulnerability_df['Overall Rank'] = vulnerability_df['Vulnerability Score'].rank()

# Sort by rank
final_ranking = vulnerability_df.sort_values('Overall Rank')[['Country', 'Warming Rate (°C/year)', 'Precipitation CV', 
                                                                'Extreme Heat Days/Year', 'Max Dry Spell (days)', 
                                                                'Overall Rank']]

print("=== CLIMATE VULNERABILITY RANKING ===")
print("(Lower Overall Rank = More Vulnerable)\n")
print(final_ranking.to_string(index=False))

In [ ]:
## COP32 Framed Findings for EthioClimate Analytics

Here are 5 evidence-backed observations for Ethiopia's COP32 position paper:

**1. Which country is warming fastest and what does the trend suggest?**

[From analysis, insert country with highest warming rate] is warming at [value]°C per year. This suggests accelerating temperature stress on agriculture, water resources, and human health. Without adaptation, crop yields could decline by [estimated percentage] by 2030.

**2. Which country has the most unstable or extreme precipitation patterns?**

[Insert country] shows the highest precipitation variability with a coefficient of variation of [value]. This means farmers cannot predict rainy seasons, leading to planting failures, reduced harvests, and food insecurity.

**3. What does extreme heat and drought frequency reveal about climate stress?**

[Insert country] experiences [value] extreme heat days per year and dry spells lasting [value] days. This combination creates compound risks: heat stress kills livestock while drought destroys crops. Communities face both threats simultaneously with no recovery time.

**4. How does Ethiopia's climate profile compare to its neighbors?**

Ethiopia ranks [insert rank] out of 5 countries in overall vulnerability. Compared to neighbors, Ethiopia has [lower/higher/similar] warming rates but [more/less] precipitation stability. Ethiopia's highlands provide some buffering, but lowland regions face severe stress similar to Sudan.

**5. Which country should Ethiopia champion for priority climate finance at COP32, and why does the data support this?**

Ethiopia should champion [insert most vulnerable country] for priority climate finance. The data shows this country has the [highest warming rate, most extreme heat days, longest dry spells, or worst precipitation variability - choose one]. Supporting this country aligns with Africa's collective bargaining position and demonstrates regional leadership.

In [ ]:
print("=== ANALYSIS COMPLETE ===")
print(f"Temperature summary saved to: ../reports/temperature_trend_comparison.png")
print(f"Precipitation boxplots saved to: ../reports/precipitation_boxplots.png")
print(f"Extreme heat chart saved to: ../reports/extreme_heat_days.png")
print(f"Dry spells chart saved to: ../reports/dry_spells.png")
print("\nVulnerability ranking completed for 5 countries.")